# 12 - Documentation RAG Backend with a Gemini Agent

This notebook follows the instructor's `backend/core.py` structure.

```text
Pinecone retriever
→ retrieve_context tool
→ content and Document artifacts
→ LangChain agent
→ run_llm()
```

Changes from the instructor are limited to Gemini-compatible model classes and the Pinecone index settings created earlier.

## 1. Optional installation

In [ ]:
# %pip install -U langchain langchain-google-genai langchain-pinecone pinecone python-dotenv truststore

## 2. SSL, imports, and environment variables

In [1]:
import truststore
truststore.inject_into_ssl()

from pathlib import Path
ENV_PATH = Path("../langchain-course/.env").resolve()

from dotenv import load_dotenv
load_dotenv(dotenv_path=ENV_PATH)

import os
from langchain.agents import create_agent
from langchain.messages import ToolMessage
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_pinecone import PineconeVectorStore

## 3. Initialize Gemini, Pinecone, and the retriever

In [2]:
INDEX_NAME = os.environ["DOCUMENTATION_INDEX_NAME"]
NAMESPACE = "langchain-docs"

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    output_dimensionality=1536,
)

llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_CHAT_MODEL", "gemini-3.5-flash-lite"),
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0,
)

vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    namespace=NAMESPACE,
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)


## 4. Create the retrieval tool

`content` is the serialized text that the agent reads. `artifact` is the original list of LangChain `Document` objects used later for source links.

In [3]:
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve LangChain documentation to help answer a query."""

    retrieved_docs = retriever.invoke(query)

    serialized = "\n\n".join(
        f"Source: {doc.metadata.get('source')}\n"
        f"Content: {doc.page_content}"
        for doc in retrieved_docs
    )

    return serialized, retrieved_docs


## 5. Create the agent

The system prompt instructs the agent to retrieve documentation before answering.

In [4]:
SYSTEM_PROMPT = """You are a LangChain documentation assistant.
Always use the retrieve_context tool before answering.
Answer using the retrieved documentation.
If the retrieved documentation does not contain the answer, say so.
"""

agent = create_agent(
    model=llm,
    tools=[retrieve_context],
    system_prompt=SYSTEM_PROMPT,
)


## 6. Create `run_llm()`

The function invokes the agent, reads the final assistant answer, and collects the raw `Document` artifacts returned by the retrieval tool.

In [5]:
def run_llm(query: str, chat_history=None):
    """Run the documentation agent and return its answer and context."""

    messages = list(chat_history or [])
    messages.append({
        "role": "user",
        "content": query,
    })

    result = agent.invoke({
        "messages": messages,
    })

    result_messages = result["messages"]
    answer = result_messages[-1].content

    context = []
    for message in result_messages:
        if (
            isinstance(message, ToolMessage)
            and message.name == "retrieve_context"
            and message.artifact
        ):
            context.extend(message.artifact)

    return {
        "answer": answer,
        "context": context,
    }


## 7. Test the backend

In [8]:
result = run_llm(
    "what are deep agents?"
)

print("Answer:")
print(result["answer"])

print("\nSources:")
for document in result["context"]:
    print("-", document.metadata.get("source"))


C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer:
[{'type': 'text', 'text': '**Deep Agents** are advanced agents built using LangChain\'s `deepagents` library designed to handle complex tasks by incorporating planning, subagents, and file system capabilities. \n\nKey features and core capabilities of deep agents include:\n\n* **Planning and Delegation:** Agents can plan tasks (e.g., using a `TodoListMiddleware` to manage TODO items with states like pending, in_progress, and completed) and delegate work to specialized **subagents**.\n* **File Systems and Execution Environments:** They can leverage file systems via backends like `StateBackend`, `StoreBackend`, and `FilesystemBackend` to read/write files and interact with structured environments.\n* **Context Management:** They support context optimization through features like:\n  * **Memory:** Maintaining persistent memory and instructions (e.g., via `AGENTS.md`).\n  * **Skills:** Dynamically loading capabilities or instructions (e.g., via `SKILL.md` sources).\n  * **Summarizat